# Step 2: Create Historical Features Baseline
This notebook calculates all the technical indicators for the entire historical dataset. It saves a `features.parquet` file which will be incrementally updated daily by the `feature_updater.py` script.

In [ ]:
import pandas as pd
import os
import sys

# Ensure phase2_qrt_challenge/scripts is in path so we can import technical_indicators
sys.path.append(os.path.abspath('phase2_qrt_challenge/scripts'))

from technical_indicators import calculate_all_indicators_parallel
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "stores"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
print('Loading top_5000_yf_data.pkl...')
pv = pd.read_pickle('top_5000_yf_data.pkl')
print('Data loaded successfully. Shape:', pv.shape)

In [ ]:
# Calculate technical indicators using parallel processing
# Note: This is computationally intensive and may take ~5-15 minutes depending on CPU
print('Calculating indicators in parallel...')
indicators_dict = calculate_all_indicators_parallel(pv, n_jobs=-1)
print('Calculation complete.')

In [ ]:
# Convert dictionary of DataFrames to a single MultiIndex DataFrame (Feature, Ticker)
indicators_df = pd.concat(indicators_dict, axis=1)

# Downcasting to float32 to save memory
indicators_df = indicators_df.astype("float32")
print('Final features shape:', indicators_df.shape)

In [ ]:
# Save the indicators to a parquet file for the daily updater and model training
output_path = os.path.join(DATA_DIR, "features.parquet")
indicators_df.to_parquet(output_path, compression="zstd", engine="pyarrow")
print(f'Saved successfully to {output_path}')